# Clinical Strand — Preprocessing & Split

**MultimodalAI‧26 · Clinical Demo**

---

Preprocess the tabular data and create stratified train/val/test splits.
The processed CSVs are the inputs to `02_train_models.ipynb`.

**Prerequisite:** `data/raw/tabular.csv` must exist — run `setup.py` first.

**Sections**
1. Setup & Load Data
2. Imputation
3. Train / Val / Test Split
4. Verify Split
5. Save Processed CSVs

---
## Section 1 — Setup & Load Data

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / 'data').exists() and (candidate / 'models').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not find clinical_strand root.')

RAW_DATA = ROOT / 'data' / 'raw' / 'tabular.csv'
PROC_DIR = ROOT / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'major_complication_30d'
print(f'ROOT: {ROOT}')

In [ ]:
df = pd.read_csv(RAW_DATA)
print(f'Shape: {df.shape}')
print(f'Complication rate: {df[TARGET].mean():.1%}')
print(f'Has notes:         {df["has_notes"].mean():.1%}')
print(f'Blood loss MNAR:   {df["blood_loss_missing"].mean():.1%}')
print()
print('Missing values (columns with any NaN):')
miss = df.isnull().sum()
print(miss[miss > 0].to_string())

In [ ]:
print('Complication rate by surgery type:')
print(df.groupby('surgery_type')[TARGET].agg(['mean', 'count']).round(3).to_string())
print()
print('Complication rate by notes availability:')
print(df.groupby('has_notes')[TARGET].agg(['mean', 'count']).round(3).to_string())

---
## Section 2 — Imputation

Model classes handle `note_risk_score` and `blood_loss_imputed` internally.
For remaining NaN values in numeric columns, apply median imputation.

> **Note:** `blood_loss_ml` NaNs are intentional MNAR — do not impute here.
> The models use `blood_loss_imputed` (pre-computed) and `blood_loss_missing` (flag).
> `note_risk_score` is also left as-is — each model handles it internally.

In [ ]:
numeric_cols = [
    'hr_mean', 'hr_std', 'rr_mean', 'rr_std',
    'spo2_mean', 'spo2_min', 'sbp_mean', 'temp_mean',
    'icu_lactate', 'icu_creatinine', 'icu_wbc', 'icu_bilirubin',
    'preop_creatinine', 'preop_wbc', 'preop_lactate',
]

to_impute = [c for c in numeric_cols if c in df.columns and df[c].isnull().sum() > 0]
for col in to_impute:
    df[col] = df[col].fillna(df[col].median())

if to_impute:
    print(f'Applied median imputation to: {to_impute}')
else:
    print('No NaN in numeric feature columns — no imputation needed.')

skip_cols = ['note_risk_score', 'blood_loss_ml']
remaining = df.drop(columns=[c for c in skip_cols if c in df.columns]).isnull().sum().sum()
print(f'Remaining NaN (excl. MNAR columns): {remaining}')

---
## Section 3 — Train / Val / Test Split

Stratified split on `major_complication_30d` to preserve class balance across folds.
For 500 patients, 70/15/15 gives roughly 350 train, 75 val, 75 test.

> **Leakage note:** imputation statistics are computed on the full dataset here for
> simplicity. A stricter approach would compute on train only then apply to val/test.
> For 500 patients the difference is negligible, but the principle matters for production.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

df_train, df_temp = train_test_split(
    df, test_size=1 - TRAIN_FRAC, stratify=df[TARGET], random_state=SEED
)
val_share = VAL_FRAC / (1 - TRAIN_FRAC)
df_val, df_test = train_test_split(
    df_temp, test_size=1 - val_share, stratify=df_temp[TARGET], random_state=SEED
)

for split, name in [(df_train, 'train'), (df_val, 'val'), (df_test, 'test')]:
    print(f'{name:6s}: n={len(split):4d}  '
          f'complication={split[TARGET].mean():.1%}  '
          f'notes={split["has_notes"].mean():.1%}')

---
## Section 4 — Verify Split

In [ ]:
print('Split verification (key subgroup rates must be consistent across splits):')
for name, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    age_b75   = (split['age'] >= 75).mean()
    bl_mnar   = split['blood_loss_missing'].mean()
    print(f'  {name:6s}: complication={split[TARGET].mean():.1%}  '
          f'age75+={age_b75:.1%}  '
          f'notes={split["has_notes"].mean():.1%}  '
          f'blood_loss_mnar={bl_mnar:.1%}')

---
## Section 5 — Save Processed CSVs

In [ ]:
df_train.to_csv(PROC_DIR / 'train.csv', index=False)
df_val.to_csv(  PROC_DIR / 'val.csv',   index=False)
df_test.to_csv( PROC_DIR / 'test.csv',  index=False)
print(f'Saved to {PROC_DIR}:')
print(f'  train.csv ({len(df_train)} rows)')
print(f'  val.csv   ({len(df_val)} rows)')
print(f'  test.csv  ({len(df_test)} rows)')
print()
print('Next: run 02_train_models.ipynb')